In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

#### Setting up project path

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "master_clean.csv"

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)

Project root: C:\dishu\Cleartrace
Data path: C:\dishu\Cleartrace\data\processed\master_clean.csv


#### Loading the dataset and validating

In [3]:
df = pd.read_csv(DATA_PATH)

df["timestamp_hour"] = pd.to_datetime(
    df["timestamp_hour"],
    errors="coerce"
)

df = (
    df.sort_values(["station_name", "timestamp_hour"])
    .reset_index(drop=True)
)

time_gaps = (
    df.groupby("station_name")["timestamp_hour"]
    .diff()
)

print("Shape:", df.shape)
print("Stations:", df["station_name"].nunique())
print("Timestamp range:", df["timestamp_hour"].min(), "to", df["timestamp_hour"].max())
print("Duplicate station-hour rows:", df.duplicated(["station_name", "timestamp_hour"]).sum())

print("\nTime gaps between consecutive rows:")
print(time_gaps.value_counts().sort_index().head(15))

Shape: (78204, 57)
Stations: 38
Timestamp range: 2026-04-10 21:00:00 to 2026-07-09 18:00:00
Duplicate station-hour rows: 0

Time gaps between consecutive rows:
timestamp_hour
0 days 01:00:00    77976
0 days 08:00:00       76
0 days 09:00:00       38
0 days 19:00:00       38
2 days 13:00:00       38
Name: count, dtype: int64


In [4]:
gap_rows = df.loc[
    time_gaps > pd.Timedelta(hours=1),
    ["station_name", "timestamp_hour"]
].copy()

gap_rows["gap_length"] = time_gaps[time_gaps > pd.Timedelta(hours=1)].values

gap_rows.head(10)

,station_name,timestamp_hour,gap_length
137,"Alipur, Delhi - DPCC",2026-04-16 22:00:00,0 days 09:00:00
515,"Alipur, Delhi - DPCC",2026-05-02 23:00:00,0 days 08:00:00
581,"Alipur, Delhi - DPCC",2026-05-06 11:00:00,0 days 19:00:00
612,"Alipur, Delhi - DPCC",2026-05-08 01:00:00,0 days 08:00:00
1725,"Alipur, Delhi - DPCC",2026-06-25 22:00:00,2 days 13:00:00
2195,"Anand Vihar, New Delhi - DPCC",2026-04-16 22:00:00,0 days 09:00:00
2573,"Anand Vihar, New Delhi - DPCC",2026-05-02 23:00:00,0 days 08:00:00
2639,"Anand Vihar, New Delhi - DPCC",2026-05-06 11:00:00,0 days 19:00:00
2670,"Anand Vihar, New Delhi - DPCC",2026-05-08 01:00:00,0 days 08:00:00
3783,"Anand Vihar, New Delhi - DPCC",2026-06-25 22:00:00,2 days 13:00:00


### Adding the target AQI 24 hours

In [5]:
df_features = df.copy()

aqi_lookup = df_features.set_index(
    ["station_name", "timestamp_hour"]
)["current_aqi"]

for hour in range(1, 25):
    future_index = pd.MultiIndex.from_arrays(
        [
            df_features["station_name"],
            df_features["timestamp_hour"] + pd.Timedelta(hours=hour)
        ],
        names=["station_name", "timestamp_hour"]
    )

    df_features[f"target_aqi_{hour}h"] = (
        aqi_lookup.reindex(future_index).to_numpy()
    )

target_columns = [
    f"target_aqi_{hour}h"
    for hour in range(1, 25)
]

print("New shape:", df_features.shape)
print("\nAvailable target values:")
print(df_features[target_columns].notna().sum())

New shape: (78204, 81)

Available target values:
target_aqi_1h     60828
target_aqi_2h     60828
target_aqi_3h     60828
target_aqi_4h     60828
target_aqi_5h     60828
target_aqi_6h     60828
target_aqi_7h     60828
target_aqi_8h     60828
target_aqi_9h     60828
target_aqi_10h    60828
target_aqi_11h    60828
target_aqi_12h    60828
target_aqi_13h    60828
target_aqi_14h    60828
target_aqi_15h    60828
target_aqi_16h    60828
target_aqi_17h    60828
target_aqi_18h    60818
target_aqi_19h    60767
target_aqi_20h    60652
target_aqi_21h    60521
target_aqi_22h    60385
target_aqi_23h    60240
target_aqi_24h    60089
dtype: int64



## Adding the time features

In [7]:
df_features["hour"] = df_features["timestamp_hour"].dt.hour
df_features["day_of_week"] = df_features["timestamp_hour"].dt.dayofweek
df_features["is_weekend"] = (
    df_features["day_of_week"] >= 5
).astype(int)
df_features["month"] = df_features["timestamp_hour"].dt.month

df_features[
    [
        "timestamp_hour",
        "hour",
        "day_of_week",
        "is_weekend",
        "month",
    ]
].head()

,timestamp_hour,hour,day_of_week,is_weekend,month
0,2026-04-10 21:00:00,21,4,0,4
1,2026-04-10 22:00:00,22,4,0,4
2,2026-04-10 23:00:00,23,4,0,4
3,2026-04-11 00:00:00,0,5,1,4
4,2026-04-11 01:00:00,1,5,1,4


#### Creating cyclic hours

In [8]:
df_features["hour_sin"] = np.sin(
    2 * np.pi * df_features["hour"] / 24
)

df_features["hour_cos"] = np.cos(
    2 * np.pi * df_features["hour"] / 24
)

df_features[
    [
        "timestamp_hour",
        "hour",
        "hour_sin",
        "hour_cos",
    ]
].head(6)



,timestamp_hour,hour,hour_sin,hour_cos
0,2026-04-10 21:00:00,21,-0.707107,0.707107
1,2026-04-10 22:00:00,22,-0.500000,0.866025
2,2026-04-10 23:00:00,23,-0.258819,0.965926
3,2026-04-11 00:00:00,0,0.000000,1.000000
4,2026-04-11 01:00:00,1,0.258819,0.965926
5,2026-04-11 02:00:00,2,0.500000,0.866025


In [9]:

df_features["day_sin"] = np.sin(
    2 * np.pi * df_features["day_of_week"] / 7
)

df_features["day_cos"] = np.cos(
    2 * np.pi * df_features["day_of_week"] / 7
)

df_features[
    [
        "timestamp_hour",
        "day_of_week",
        "day_sin",
        "day_cos",
    ]
].head(8)

,timestamp_hour,day_of_week,day_sin,day_cos
0,2026-04-10 21:00:00,4,-0.433884,-0.900969
1,2026-04-10 22:00:00,4,-0.433884,-0.900969
2,2026-04-10 23:00:00,4,-0.433884,-0.900969
3,2026-04-11 00:00:00,5,-0.974928,-0.222521
4,2026-04-11 01:00:00,5,-0.974928,-0.222521
5,2026-04-11 02:00:00,5,-0.974928,-0.222521
6,2026-04-11 03:00:00,5,-0.974928,-0.222521
7,2026-04-11 04:00:00,5,-0.974928,-0.222521


## Creating AQI lag features

In [10]:
aqi_lookup = df_features.set_index(
    ["station_name", "timestamp_hour"]
)["current_aqi"]

lag_hours = [1, 6, 12, 24]

for hour in lag_hours:
    past_index = pd.MultiIndex.from_arrays(
        [
            df_features["station_name"],
            df_features["timestamp_hour"] - pd.Timedelta(hours=hour)
        ],
        names=["station_name", "timestamp_hour"]
    )

    df_features[f"aqi_lag_{hour}h"] = (
        aqi_lookup.reindex(past_index).to_numpy()
    )

lag_columns = [f"aqi_lag_{hour}h" for hour in lag_hours]

print(df_features[lag_columns].notna().sum())

aqi_lag_1h     60662
aqi_lag_6h     59815
aqi_lag_12h    59264
aqi_lag_24h    58368
dtype: int64


## v1 limitation — mixed-provenance pollutant features

The pollutant columns used below may contain original observations, short-gap interpolation and neighbouring-station proxy estimates. Consequently, these pollutant lag and rolling features belong only to the original v1 pipeline and are not part of the final modelling contract.

Notebook 04 reconstructs `<pollutant>_observed` using the original `has_<pollutant>` flags and rebuilds all pollutant lags and rolling statistics from observations only. The resulting `features_core_v2.csv` is the authoritative modelling dataset.

### Creating lag features for pollutants

In [11]:
pollutant_lag_config = {
    "pm25": [1, 6, 12, 24],
    "pm10": [1, 6, 12, 24],
    "no2": [1, 6, 24],
    "co": [1, 6, 24],
    "o3": [1, 6, 24],
    "so2": [1, 6],
}

pollutant_lag_columns = []

for pollutant, lag_hours in pollutant_lag_config.items():
    pollutant_lookup = df_features.set_index(
        ["station_name", "timestamp_hour"]
    )[pollutant]

    for hour in lag_hours:
        past_index = pd.MultiIndex.from_arrays(
            [
                df_features["station_name"],
                df_features["timestamp_hour"] - pd.Timedelta(hours=hour),
            ],
            names=["station_name", "timestamp_hour"],
        )

        column_name = f"{pollutant}_lag_{hour}h"

        df_features[column_name] = (
            pollutant_lookup.reindex(past_index).to_numpy()
        )

        pollutant_lag_columns.append(column_name)

print("Pollutant lag columns created:", len(pollutant_lag_columns))
print(df_features[pollutant_lag_columns].notna().sum())

Pollutant lag columns created: 19
pm25_lag_1h     77969
pm25_lag_6h     76829
pm25_lag_12h    75993
pm25_lag_24h    74853
pm10_lag_1h     77969
pm10_lag_6h     76829
pm10_lag_12h    75993
pm10_lag_24h    74853
no2_lag_1h      77963
no2_lag_6h      76823
no2_lag_24h     74852
co_lag_1h       77963
co_lag_6h       76823
co_lag_24h      74852
o3_lag_1h       77969
o3_lag_6h       76829
o3_lag_24h      74853
so2_lag_1h      77885
so2_lag_6h      76745
dtype: int64


### Creating rolling features

In [14]:
rolling_config = {
    "pm25": [6, 12, 24],
    "pm10": [6, 12, 24],
    "no2": [6, 12],
    "co": [6, 12],
    "o3": [6, 12],
    "so2": [6],
}

rolling_columns = []

for pollutant, windows in rolling_config.items():
    for window in windows:
        column_name = f"{pollutant}_rolling_mean_{window}h"
        df_features[column_name] = np.nan

        for _, station_rows in df_features.groupby("station_name").groups.items():
            station_data = (
                df_features.loc[station_rows, ["timestamp_hour", pollutant]]
                .sort_values("timestamp_hour")
            )

            rolling_values = station_data.rolling(
                window=f"{window}h",
                on="timestamp_hour",
                min_periods=max(2, window // 2),
            )[pollutant].mean()

            df_features.loc[
                station_data.index,
                column_name
            ] = rolling_values.to_numpy()

        rolling_columns.append(column_name)

print("Rolling features created:", len(rolling_columns))
print(df_features[rolling_columns].notna().sum())

Rolling features created: 13
pm25_rolling_mean_6h     77741
pm25_rolling_mean_12h    77057
pm25_rolling_mean_24h    76948
pm10_rolling_mean_6h     77741
pm10_rolling_mean_12h    77057
pm10_rolling_mean_24h    76948
no2_rolling_mean_6h      77736
no2_rolling_mean_12h     77057
co_rolling_mean_6h       77736
co_rolling_mean_12h      77057
o3_rolling_mean_6h       77741
o3_rolling_mean_12h      77057
so2_rolling_mean_6h      77658
dtype: int64


In [15]:
weather_keywords = [
    "temperature",
    "humidity",
    "wind",
    "pressure",
    "precipitation",
    "rain",
    "cloud",
]

weather_columns = [
    column
    for column in df_features.columns
    if any(keyword in column.lower() for keyword in weather_keywords)
]

weather_columns

['temperature_2m',
 'relative_humidity_2m',
 'precipitation',
 'surface_pressure',
 'wind_speed_10m',
 'wind_direction_10m']

### Converting wind speed and direction into spatial components

In [16]:
wind_angle = np.deg2rad(df_features["wind_direction_10m"])

df_features["wind_u"] = (
    -df_features["wind_speed_10m"] * np.sin(wind_angle)
)

df_features["wind_v"] = (
    -df_features["wind_speed_10m"] * np.cos(wind_angle)
)

df_features["is_raining"] = (
    df_features["precipitation"] > 0
).astype(int)

df_features[
    [
        "wind_speed_10m",
        "wind_direction_10m",
        "wind_u",
        "wind_v",
        "precipitation",
        "is_raining",
    ]
].head()

,wind_speed_10m,wind_direction_10m,wind_u,wind_v,precipitation,is_raining
0,15.3,289.0,14.466434,-4.981193,0.0,0
1,14.8,297.0,13.186897,-6.719059,0.0,0
2,13.4,299.0,11.719904,-6.496449,0.0,0
3,13.9,304.0,11.523622,-7.772781,0.0,0
4,14.2,309.0,11.035473,-8.936350,0.0,0


## Weather-Derived Features

Convert wind speed from km/h to m/s and decompose meteorological wind direction into east–west (`wind_u_ms`) and north–south (`wind_v_ms`) components. A binary rainfall indicator is also created from hourly precipitation.

In [17]:
df_features.drop(
    columns=["wind_u", "wind_v"],
    errors="ignore",
    inplace=True,
)

df_features["wind_speed_10m_ms"] = (
    df_features["wind_speed_10m"] / 3.6
)

wind_angle = np.deg2rad(
    df_features["wind_direction_10m"]
)

df_features["wind_u_ms"] = (
    -df_features["wind_speed_10m_ms"]
    * np.sin(wind_angle)
)

df_features["wind_v_ms"] = (
    -df_features["wind_speed_10m_ms"]
    * np.cos(wind_angle)
)

df_features["is_raining"] = (
    df_features["precipitation"] > 0
).astype(int)

df_features[
    [
        "wind_speed_10m",
        "wind_speed_10m_ms",
        "wind_direction_10m",
        "wind_u_ms",
        "wind_v_ms",
        "precipitation",
        "is_raining",
    ]
].head()

,wind_speed_10m,wind_speed_10m_ms,wind_direction_10m,wind_u_ms,wind_v_ms,precipitation,is_raining
0,15.3,4.250000,289.0,4.018454,-1.383665,0.0,0
1,14.8,4.111111,297.0,3.663027,-1.866405,0.0,0
2,13.4,3.722222,299.0,3.255529,-1.804569,0.0,0
3,13.9,3.861111,304.0,3.201006,-2.159106,0.0,0
4,14.2,3.944444,309.0,3.065409,-2.482319,0.0,0


In [21]:
engineered_columns = (
    target_columns
    + ["hour", "day_of_week", "is_weekend", "month",
       "hour_sin", "hour_cos", "day_sin", "day_cos"]
    + lag_columns
    + pollutant_lag_columns
    + rolling_columns
    + ["wind_speed_10m_ms", "wind_u_ms", "wind_v_ms", "is_raining"]
)

print("Final shape:", df_features.shape)
print(
    "Duplicate station-hour rows:",
    df_features.duplicated(["station_name", "timestamp_hour"]).sum()
)
print(
    "Infinite values:",
    np.isinf(df_features.select_dtypes(include=np.number)).sum().sum()
)

print("\nEngineered feature count:", len(engineered_columns))
print("\nMissing percentage in engineered columns:")
print(
    df_features[engineered_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .head(15)
)

Final shape: (78204, 129)
Duplicate station-hour rows: 0
Infinite values: 0

Engineered feature count: 72

Missing percentage in engineered columns:
aqi_lag_24h       25.364431
aqi_lag_12h       24.218710
aqi_lag_6h        23.514142
target_aqi_24h    23.163777
target_aqi_23h    22.970692
target_aqi_22h    22.785280
target_aqi_21h    22.611375
target_aqi_20h    22.443865
aqi_lag_1h        22.431078
target_aqi_19h    22.296813
target_aqi_18h    22.231599
target_aqi_5h     22.218812
target_aqi_4h     22.218812
target_aqi_3h     22.218812
target_aqi_2h     22.218812
dtype: float64


In [24]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "features_core_v1.csv"
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_features.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved to:", OUTPUT_PATH)
print("Exported shape:", df_features.shape)

Saved to: C:\dishu\Cleartrace\data\features\features_core_v1.csv
Exported shape: (78204, 129)
